Build a nano GPT Part-02
-----
[video](https://youtu.be/kCc8FmEb1nY?t=2533)

[Attention-is-all-you-need](https://arxiv.org/pdf/1706.03762)

<ins>Goal</ins>

## Attention and Self-Attention Intro


**Attention** in neural networks is a mechanism that allows the model to focus on the most relevant parts of the input data when processing information. Instead of treating all parts of the input equally, attention assigns different weights to different elements, highlighting those that are most important for the current task.

* Self-Attention
  * A mechanism where a sequence attends to **itself**.
  * The Query, Key, and Value are all from the same sequence.
  * Helps the model understand relationships *within* a single sequence.

* Cross-Attention
  * A mechanism where one sequence attends to **another sequence**.
  * The Query comes from one sequence, while the Key and Value come from a different sequence.
  * Helps the model relate information between two different sequences.

Both self-attention and cross-attention are key components of the Transformer architecture, enabling powerful sequence processing capabilities.

Self-attention allows a model to weigh the importance of different words/char (or tokens) in a sequence relative to each other. It computes a weighted sum of all input embeddings, where the weights are determined by how relevant each word is to the others

**Key Points of Self-Attention:**

*   **Contextual Understanding:** It helps the model understand the meaning of a token based on its relationship with other tokens in the same sequence.
*   **Parallel Processing:** Unlike traditional sequential models (like RNNs), self-attention can process all tokens in a sequence in parallel, making it more efficient for long sequences.
*   **Variable Dependencies:** It can capture long-range dependencies between tokens, regardless of their distance in the sequence.
*   **Learned Weights:** The importance given to each token is learned during the training process.
*   **Core of Transformers:** Self-attention is a fundamental component of the Transformer architecture, which has achieved state-of-the-art results in various NLP tasks, including character-level language modeling.

**What is Bag-of-Words?**

Bag-of-words is a much simpler way to represent text. It just counts how many times each word appears in a document. It completely ignores the order of the words and the relationships between them.

For example, the sentences "The quick brown fox jumps over the lazy dog" and "The lazy dog jumps over the quick brown fox" would have the same bag-of-words representation because they use the same words the same number of times, even though their meaning is different.

**Is Bag-of-Words used in self-attention?**

No, bag-of-words is **not** used in self-attention. Self-attention, as explained above, specifically *does* consider the relationships and order of tokens in the sequence through the Query, Key, and Value mechanism. This is a key difference that makes self-attention (and models like transformers that use it) much more powerful for understanding context and generating coherent text compared to models based on bag-of-words.


**Simplified working of self-attention**

let's use the sentence "The dog ate the bone because it was hungry." to explain the point-by-point breakdown of self-attention. In this sentence, "it" refers to "dog".

1.  **Looking at Each Token:** The computer looks at each token: "The", "dog", "ate", "the", "bone", "because", "it", "was", "hungry", "."

2.  **Asking Questions (Queries), Getting Information (Keys), and Having the Content (Values):** For *each* of these ten tokens, the computer creates a Query, a Key, and a Value. Think of them like this for the token "dog":
    *   **Query ("dog"):** "What information about other tokens is relevant to understanding 'dog'?"
    *   **Key ("dog"):** "Here is the information I can offer about 'dog'."
    *   **Value ("dog"):** The actual meaning or representation of the token "dog".

    Let's look at the token "it" as another example:
    *   **Query ("it"):** "What information about other tokens is relevant to understanding 'it'?"
    *   **Key ("it"):** "Here is the information I can offer about 'it'."
    *   **Value ("it"):** The actual meaning or representation of the token "it".

3.  **Comparing and Scoring:** This is where the magic happens. The computer takes the Query of *one* token (let's say, the Query for "it") and compares it to the Key of *every other token* in the sentence.
    *   Query ("it") vs. Key ("The") -> Score 1
    *   Query ("it") vs. Key ("dog") -> Score 2 (This score will be high!)
    *   Query ("it") vs. Key ("ate") -> Score 3
    *   Query ("it") vs. Key ("the") -> Score 4
    *   Query ("it") vs. Key ("bone") -> Score 5
    *   Query ("it") vs. Key ("because") -> Score 6
    *   Query ("it") vs. Key ("it") -> Score 7 (This score will also be high!)
    *   Query ("it") vs. Key ("was") -> Score 8
    *   Query ("it") vs. Key ("hungry") -> Score 9
    *   Query ("it") vs. Key (".") -> Score 10

    This comparison is done for the Query of *every* token against the Key of *every* other token.

4.  **Weighting the Importance:** The scores from the comparison step are converted into weights. For "it", the weight associated with "dog" will be significantly higher than the weights for most other words because the comparison score was high.

5.  **Combining Information:** To create the new, context-aware representation for "it", the computer takes the Value of *every* token in the sentence and adds them up, but each Value is multiplied by the attention weight calculated in the previous step. Since the weight for "dog" was high when considering "it", the Value of "dog" will contribute a lot to the final representation of "it". This is how the model learns that "it" in this sentence refers to "dog".

This same process of comparing Queries and Keys, getting scores, converting to weights, and combining weighted Values is done for every single token in the input sequence simultaneously. This allows the model to build a rich understanding of how all the tokens relate to each other within their context.

###**Optional: The mathematical trick used in self-attention**
Here we show 3 diff ways to calculate weighted average of previous char. We will use one of this option(ideally option3) to calculate weighted average in our self-attention impleenation


In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
# consider the following toy example:
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [2]:
# option 1: finding weighted average/mean of past chars - using for loop
xbow = torch.zeros(B,T,C)
for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1]
    xbow[b,t] = torch.mean(xprev, 0)
print(x[0])
print(xbow[0])

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [3]:
# option 2: finding weighted average of past chars - using matrix multiplication
wei = torch.tril(torch.ones(T,T)) # creates a lower triangular matrix of ones
sum = wei.sum(1, keepdim=True) # calculates the sum of each row
wei = wei / wei.sum(1, keepdim=True) # normalizes the wei matrix by dividing each element by its row sum

# calculates the weighted average of past characters for each position in the sequence across all batches
xbow2 = wei @ x # here wei.shape is [T,T] and x.shape is [B,T,C] , so during matrix multiplication wei will be broadcasted to [B,T,T]
torch.allclose(xbow, xbow2) # ideally this should be true, but might show false due to differences in floating-point precision

False

In [4]:
# option 3: finding weighted average of past chars - using softmax
tril = torch.tril(torch.ones(T, T))# creates a lower triangular matrix of ones
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf')) # masked_fill tells set inf to all elements in tril where its value is 0
wei = F.softmax(wei, dim=-1) # softmax exponentiate each element and divide by sum
xbow3 = wei @ x
torch.allclose(xbow, xbow3)# ideally this should be true, but might show false due to differences in floating-point precision

False

## Self-attention core
[video](https://youtu.be/kCc8FmEb1nY?t=3720)

This code implements a single "Self-Attention Head" (specifically a decoder-style self-attention) which is a core component of the Transformer architecture. It shows how a token in a sequence pays attention to other tokens in the same sequence to build a richer representation

**decoder-style self-attention:** is an attention mechanism where a token is only allowed to attend to previous tokens and itself

In [5]:
# single self-attention(decoder-style self-attention) sample code
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)
head_size = 16 # defines the dimensionality of the Query, Key, and Value vectors for this single self-attention head

## linear layers for key, query and value##
key = nn.Linear(C, head_size, bias=False) #creating linear layers with fixed weight and no biases -  this layer transform input features into a Query representation
query = nn.Linear(C, head_size, bias=False) #creating linear layers with fixed weight and no biases -  this layer transform input features into a Key representation
value = nn.Linear(C, head_size, bias=False) #creating linear layers with fixed weight and no biases -  this layer transform input features into a Value representation

## As per self-attention - generating the key, queries and values for the input x ##
k = key(x)   # forwarding through layer key - it does matrix multi(x @ key.weight.T) and result will be (B, T, 16)
q = query(x) # forwarding through layer query - it does matrix multi(x @ query.weight.T) and result will be (B, T, 16)
v = value(x) # forwarding through layer value - it does matrix multi(x @ value.weight.T) and result will be (B, T, 16)

## Comparing and scoring as per self-attention. ie comparison is done for the Query of every token against the Key of every other token.##
# The result is the raw affinities(Attention Scores) matrix: which is the attention scores between every pair of tokens in the sequence.
# A higher score means the i-th token finds the j-th token more relevant
# matrix within the batch shows the attention scores between every pair of tokens in the sequence
#wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T) . Here we need to transpose for matrix multiplation to happen
wei = q @ k.transpose(-2, -1) * head_size**-0.5 # the "*head_size**-0.5" is done to Prevents Exploding Gradients and to stabilize softmax output

## Scoring and Weighting as per self-attention ##
tril = torch.tril(torch.ones(T, T)) # This mask is crucial for decoder-style self-attention, where a token is only allowed to attend to previous tokens and itself
wei = wei.masked_fill(tril == 0, float('-inf')) # masked_fill tells to set 'inf' to all elements in tril where its value is 0
wei = F.softmax(wei, dim=-1) # softmax exponentiate each element and divide by sum. This converts the raw attention scores into a probability distribution where the values in each row sum up to 1

## Combining Information as per self-attention. ##
# Each token's representation in 'out' is now a combination of the Value features of all tokens it attended to, weighted by the attention scores.
out = wei @ v # this matrix multiplcation performs a weighted sum of the Value features for each token, where the weights are given by the attention scores.
out.shape

torch.Size([4, 8, 16])

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

**What's the impact of the dimension reduction in the attention head on the next layer?**

The reduction in dimension from (4, 8, 32) (your input x) to (4, 8, 16) (the output of the single self-attention head, where head_size=16) does have an impact. If the dimension is significantly reduced and the model architecture doesn't compensate elsewhere (like having enough heads in Multi-Head Attention or sufficiently large subsequent layers), the attention layer could become an "information bottleneck." It might not be able to pass all the necessary contextual information to the next layers

One solution: In a standard Multi-Head Attention layer, you would typically have multiple heads (say, num_heads) working in parallel. Each head would have a head_size. The outputs of these heads, each with shape (B, T, head_size), are then concatenated along the last dimension. This results in a tensor with shape (B, T, num_heads * head_size).Often, num_heads * head_size is designed to be equal to the original input embedding dimension C

**Why we need 'Value' layer when we already have value in input ?**

You have the original information about each token in x.

* The Query layer creates a version of that information that's good for asking questions about other tokens.
* The Key layer creates a version of that information that's good for answering questions from other tokens.
* The Value layer creates a version of that information that's best suited for being combined or mixed together once the attention weights are calculated.

So, even though x has the initial information, the value layer transforms that information into a format that's specifically optimized for the final step where the attention weights are used to blend the information from different tokens. It's like preparing the ingredients (the Values) in the best way before mixing them together (the weighted sum).

**Why do we need postional encoding when we already have attention mechanism?**

Short answer: We need positional encoding because the attention mechanism alone does not track word order, and meaning in language (or any sequential data) heavily depends on position


The standard self-attention mechanism, by itself, is permutation-invariant.
What does that mean? It means if you were to randomly shuffle the order of the tokens in the input sequence, the self-attention calculation would still produce the same attention scores and the same output representations for each token (though the order of the output tokens would also be shuffled).

Self-attention calculates relationships based on the dot product of Queries and Keys, which only depends on the content of the token vectors, not their position in the sequence.

However, for tasks like language modeling, the order of words is absolutely crucial for understanding meaning. "Dog bites man" has a very different meaning than "Man bites dog," even though they use the same words.

This is where position encoding comes in. We add positional information to the token embeddings before they go into the self-attention layers. This way, the self-attention mechanism, when it calculates the compatibility between Query and Key vectors, is now working with vectors that contain both the token's content and its position in the sequence

**So, to summarize:**

  * Self-attention finds relationships based on content.
  * Position encoding provides information about order.

You need both: self-attention to find complex relationships based on the enriched embeddings, and position encoding to ensure those relationships are understood in the context of the sequence's order. Without position encoding, the model wouldn't know the difference between "Dog bites man" and "Man bites dog."

## Sample model with position encoding,multi head self-attention & feedforward layer

**why do we need feed forwarding layer?**

In standard Transformer block, a FeedForward layer is typically added after the attention mechanism. The main reason for adding the FeedForward layer is to further process the context-aware representations generated by the attention layer and introduce non-linearity

While attention allows tokens to communicate and aggregate information, the FeedForward layer processes this aggregated information to extract higher-level features and patterns.

**What is Transformer Block ?**

A Block represents a single layer of the core Transformer processing. It combines the two main mechanisms: communication (Attention) and computation (FeedForward).

**Why Multiple Blocks used in model?**

Stacking multiple blocks sequentially allows the model to iteratively.Each block can learn to capture increasingly complex patterns and dependencies. The first block might learn simple relationships, while subsequent blocks can build upon those to understand more abstract or long-range connections. Basically, Stacking blocks increases the overall capacity of the model to learn complex functions and understand the data

**The choice of how many blocks to use?**

There's no single magic number. The optimal number of blocks depends on the complexity of the task, the size of the dataset, and the computational resources available. More blocks generally mean higher capacity but also more computation and a higher risk of overfitting if the dataset is small.

In [7]:
# download the sample dataset(tiny shakespeare dataset)
!curl https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -o input.txt -s

In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32 # how many independent sequences will we process in parallel?
ctx_len = 8 # what is the maximum context length for predictions?
max_iters = 5000 # training iteration
eval_interval = 500 # interval at which we evaluate the loss on train and val sets during training
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32 # dim of input embedding

torch.manual_seed(1337) # ensures random number generation is reproducible

# read the input file
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# construct vocab
chars = sorted (list(set(text))) # extract all unique chars from entire text
vocab_size = len(chars)

# construct encoder and decoder
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[ch] for ch in s] #encoder: take a string, output a list of integers
decode = lambda ix: ''.join([itos[i] for i in ix]) # decoder: take a list of integers, output a string

# tokenization/encode the entire text to integer ids
data = torch.tensor(encode(text), dtype=torch.long)

## Build training & validation split from encoded data ##
n = int(0.9 * len(data)) # first 90% will be used for train and remaining 10% for validation
train_data = data[:n]
val_data = data[n:]

# method to generate random batch of inputs and target data.
def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - ctx_len, (batch_size,)) # generates batch_size (4) random integers.
  # generate input and output sequence based on above 4 random integers
  x = torch.stack([data[i:i+ctx_len] for i in ix]) # rows of input sequence
  y = torch.stack([data[i+1:i+ctx_len+1] for i in ix]) # rows of output sequence
  x, y = x.to(device), y.to(device) # use cuba
  return x, y

# function helps you periodically assess how well your model is performing on training & validation data
# This is crucial for monitoring training progress and detecting overfitting
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean() # Calculates the average loss over all iterations for the current split ('train' or 'val')
    model.train()
    return out

class Head(nn.Module):
      """ one head of self-attention """

      def __init__(self, head_size):
          super().__init__()
          #init query,key and value
          self.key = nn.Linear(n_embd, head_size, bias=False)
          self.query = nn.Linear(n_embd, head_size, bias=False)
          self.value = nn.Linear(n_embd, head_size, bias=False)
          # creates a lower triangular matrix of ones - register it as buffer as they are not trainable parameters
          self.register_buffer('tril', torch.tril(torch.ones(ctx_len, ctx_len)))

      def forward(self, x):
          B,T,C = x.shape
          # forward through query,key and value
          k = self.key(x)   # (B,T,head_size)
          q = self.query(x) # (B,T,head_size)
          v = self.value(x) # (B,T,head_size)
          # compute attention scores ("affinities")
          wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
          wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
          wei = F.softmax(wei, dim=-1) # (B, T, T)
          # perform the weighted aggregation of the values
          out = wei @ v # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
          return out

class MultiHeadAttention(nn.Module):
      """ multiple heads of self-attention in parallel """

      def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

      def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1) #concat the last dim

class FeedForward(nn.Module):
      """ a simple linear layer followed by a non-linearity """

      def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
          nn.Linear(n_embd, n_embd),
          nn.ReLU(),
        )

      def forward(self, x):
        return self.net(x)

class Block(nn.Module):
      """ Transformer block: communication followed by computation """

      def __init__(self, n_embd, n_head):
          # n_embd: embedding dimension, n_head: the number of heads we'd like
          super().__init__()
          head_size = n_embd // n_head # 32/4 --> 8
          # the communcation
          self.sa = MultiHeadAttention(n_head, head_size)
          # the computation
          self.ffwd = FeedForward(n_embd)

      def forward(self, x):
          x = x + self.sa(x)
          x = x + self.ffwd(x)
          return x

# simple Pytorch Bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) # input embedding layer
        self.position_embedding_table = nn.Embedding(ctx_len, n_embd) # position embedding layer
        # self.sa_head = MultiHeadAttention(4, n_embd//4) # so in total, 4 heads of 8-Dim self-attentions
        # self.ffwd = FeedForward(n_embd)
        self.blocks = nn.Sequential( # why is that we use 3 block?
          Block(n_embd, n_head=4),
          Block(n_embd, n_head=4),
          Block(n_embd, n_head=4),
        )
        self.lm_head = nn.Linear(n_embd, vocab_size) # hidden layer. lm_head means language modeling head

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B-Batch,T-Times,C-Channel)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        ## feeding for self-attention and its compuation
        # x = self.sa_head(x) #(B,T, num_heads * head_size)
        # x = self.ffwd(x)
        x = self.blocks(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # reshaping to conform with pytorch cross_entropy. i.e [4,8,65] to [32,65]
            targets = targets.view(B*T) # reshaping to conform with pytorch cross_entropy. i.e [4,8] to [32]
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size token bcoz the model model predicts one new token at a time and appends it to the idx sequence. As new tokens are added, the idx sequence grows longer than ctx_len
            idx_cond = idx[:, -ctx_len:]

            logits, loss = self(idx_cond) # calls 'forward' method to get the predictions. 'loss' can be ignored
            # focus only on the last time(T) step. -1 extracts the last element along the specified dimension(here its time dim 'T')
            logits = logits[:, -1, :] #  becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)

            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)

            # concatenates the newly sampled token IDs (idx_next) to the end of the current sequence (idx) along the time dimension (dim=1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1) . T dimension size increase over the iteration.
        return idx

# init the model
model = BigramLanguageModel()
m = model.to(device)

# create a PyTorch optimizer - here we using AdamW instead of SGD optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

# train the model
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train') # sample a batch of data

    logits, loss = model(xb, yb) # evaluate the loss
    optimizer.zero_grad(set_to_none=True) # zero grad
    loss.backward() # backpass
    optimizer.step() # update parameters
print(loss.item())

#lets see what model can generate after training
context = torch.zeros((1, 1), dtype=torch.long, device=device) # basically, 0 is char '\n'. we start with newline to predict the next char and so on
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

step 0: train loss 5.1768, val loss 5.1703
step 500: train loss 2.5026, val loss 2.5285
step 1000: train loss 2.3567, val loss 2.3676
step 1500: train loss 2.2865, val loss 2.3115
step 2000: train loss 2.2227, val loss 2.2709
step 2500: train loss 2.1990, val loss 2.2268
step 3000: train loss 2.1612, val loss 2.2154
step 3500: train loss 2.1561, val loss 2.1986
step 4000: train loss 2.1427, val loss 2.1778
step 4500: train loss 2.1121, val loss 2.1753
2.0259008407592773

thase he garnewe than havis youfe, eflardel what rive annd no apis corbferghou the kasay:
The vairls you, this falt do wode!
And all his gre! humom her,
And dond as ounf, this her
Wo.

MOIES:
Tame I ruce! a with; our xod,
Tha sut
Rechan;
Ont dattengh's mot sele the the pad him:
He valaws would him bay in so mosck'd's bruse sayel, noisuach
plos it whougail and and wee erive makin poor I bery so to tromes!
Be his dave to
agryou, would lordend tarrre are brime commanas hearsbon mosth theall veay sh
